In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

df = pd.read_csv("../../../data/insurance.csv")

X = df.drop("charges", axis=1)
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_features = ["age", "bmi", "children"]
cat_features = ["sex", "smoker", "region"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

rf = RandomForestRegressor(random_state=42)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", rf)
])

param_grid = {
    "regressor__n_estimators": [100, 200, 300],
    "regressor__max_depth": [None, 5, 10, 20],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    model,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best params:", grid.best_params_)

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2:   {r2:.4f}")

Best params: {'regressor__max_depth': 5, 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 200}
MAE:  2488.02
RMSE: 4343.17
R2:   0.8785


In [2]:
import wandb

run = wandb.init(
    project="Medical-Insurance-Cost-Prediction", 
    name="RandomForestRegressor", 
    config={
        "model": "RandomForestRegressor",
        "C": 1.0,
        "cv_folds": 5,
        "max_iter": 1000,
        "test_size": 0.2,
        "random_state": 42
    }
)

print(run.config)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ngocdo5852 (ngocdo5852-dai-hoc-mo). Use `wandb login --relogin` to force relogin


{'model': 'RandomForestRegressor', 'C': 1.0, 'cv_folds': 5, 'max_iter': 1000, 'test_size': 0.2, 'random_state': 42}


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.5, color='blue')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Đường chuẩn y=x
ax.set_xlabel('Chi phí thực tế (Actual)')
ax.set_ylabel('Chi phí dự đoán (Predicted)')
ax.set_title('Biểu đồ Thực tế vs Dự đoán - SVR')

wandb.log({
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "Actual_vs_Predicted": wandb.Image(fig)
})

plt.close(fig)